In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib
import os

In [2]:
X_test = pd.read_csv("../data/X_test.csv")
y_test = pd.read_csv("../data/y_test.csv")["Attrition"]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTarget distribution:")
print(y_test.value_counts())

X_test shape: (294, 52)
y_test shape: (294,)

Target distribution:
Attrition
0    247
1     47
Name: count, dtype: int64


In [3]:
y_test = pd.read_csv("../data/y_test.csv")["Attrition"]

In [4]:
import os

print(os.listdir("../data"))

['X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']


In [5]:
X_train = pd.read_csv("../data/X_train.csv")
y_train = pd.read_csv("../data/y_train.csv")["Attrition"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1176, 52)
y_train shape: (1176,)


In [6]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 52
Number of categorical features: 0

Numerical features:
['Age', 'HourlyRate', 'Education', 'DistanceFromHome', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'DailyRate', 'MonthlyRate', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Gender_F', 'Gender_M', 'MaritalStatus_Divorced', 'MaritalStatus_Married', 'MaritalStatus_Single', 'Department_Human Resources', 'Department_Research & Development', 'Department_Sales', 'JobRole_Healthcare Representative', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRole_Sales Representative

In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [8]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

print("Baseline Logistic Regression model created.")

Baseline Logistic Regression model created.


In [9]:
baseline_model.fit(X_train, y_train)

print("Baseline model training completed successfully.")


Baseline model training completed successfully.


In [10]:
y_pred = baseline_model.predict(X_test)

y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))

Predictions generated successfully.
Number of predictions: 294


In [11]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("BASELINE MODEL PERFORMANCE")
print("===========================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


BASELINE MODEL PERFORMANCE
Accuracy : 0.8605
Precision: 0.6154
Recall   : 0.3404
F1 Score : 0.4384
ROC-AUC  : 0.8080


In [12]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.62      0.34      0.44        47

    accuracy                           0.86       294
   macro avg       0.75      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294


Confusion Matrix:
[[237  10]
 [ 31  16]]


In [13]:
import os
import joblib
import pandas as pd

# Create folders if they don't exist
os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

# Save trained model
joblib.dump(
    baseline_model,
    "../models/baseline_logistic_regression.pkl"
)

# Save evaluation metrics
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results.to_csv(
    "../outputs/baseline_model_metrics.csv",
    index=False
)

# Save predictions
predictions = pd.DataFrame({
    "Actual_Attrition": y_test.values,
    "Predicted_Attrition": y_pred,
    "Attrition_Probability": y_prob
})

predictions.to_csv(
    "../outputs/baseline_predictions.csv",
    index=False
)

print("Baseline model saved successfully.")
print("Metrics saved successfully.")
print("Predictions saved successfully.")

Baseline model saved successfully.
Metrics saved successfully.
Predictions saved successfully.


# Milestone 2 – Baseline Model Development and Evaluation

## Assigned Task

Develop and evaluate a baseline machine learning model for predicting employee attrition using the provided training and testing datasets.

## Objective

The objective of this task is to establish a baseline classification model that can predict whether an employee is likely to leave the organization based on the available employee-related features.

## Dataset

The dataset was divided into training and testing sets.

* Training features: **1,176 records × 52 features**
* Training target: **1,176 records**
* Testing features: **294 records × 52 features**
* Testing target: **294 records**
* Target variable: **Attrition**
* Target classes:

  * `0` – No Attrition
  * `1` – Attrition

The test dataset contains **247 non-attrition cases** and **47 attrition cases**, indicating class imbalance in the target variable.

## Data Preprocessing

The following preprocessing steps were implemented:

* Numerical and categorical features were identified automatically.
* Missing numerical values were handled using median imputation.
* Numerical features were standardized using `StandardScaler`.
* Missing categorical values were handled using the most frequent value.
* Categorical variables were converted into numerical representations using `OneHotEncoder`.
* `handle_unknown="ignore"` was used to safely process categorical values that may appear in the test dataset but not in the training dataset.

A `ColumnTransformer` was used to apply the appropriate preprocessing to numerical and categorical features.

## Baseline Model

A **Logistic Regression** classifier was selected as the baseline machine learning model.

The preprocessing and classification steps were combined into a single Scikit-learn pipeline.

The model was trained using the training dataset and evaluated using the unseen test dataset.

## Model Evaluation

The following evaluation metrics were calculated:

* Accuracy
* Precision
* Recall
* F1 Score
* ROC-AUC
* Classification Report
* Confusion Matrix

## Baseline Model Results

| Metric    |      Score |
| --------- | ---------: |
| Accuracy  | **86.05%** |
| Precision | **61.54%** |
| Recall    | **34.04%** |
| F1 Score  | **43.84%** |
| ROC-AUC   | **80.80%** |

## Confusion Matrix

```text
[[237, 10],
 [31, 16]]
```

The model correctly classified **237 non-attrition cases** and **16 attrition cases**. It incorrectly classified **10 non-attrition cases as attrition** and missed **31 actual attrition cases**.

## Baseline Model Analysis

The baseline model achieved an accuracy of **86.05%** and a ROC-AUC score of **80.80%**.

However, the recall for the attrition class was **34.04%**, indicating that the baseline model was able to identify only a portion of the employees who actually experienced attrition.

This result provides a useful baseline for future model improvement, particularly for improving the identification of attrition cases while maintaining an acceptable level of precision.

## Output Files

The following outputs were generated:

* `baseline_logistic_regression.pkl` – trained baseline model
* `baseline_model_metrics.csv` – evaluation metrics
* `baseline_predictions.csv` – actual values, predicted values, and attrition probabilities

## Conclusion

A complete baseline machine learning pipeline was successfully developed for employee attrition prediction. The workflow includes data loading, preprocessing, model training, prediction, evaluation, and saving of the trained model and evaluation outputs.

The baseline results can be used as a reference point for future model improvement and comparison with more advanced classification techniques.
